# Does a smaller, more stable panel transfer as well?

Boruta reselected the 30 panel genes at very different rates across the 25 training folds: *RBM4B* in every
fold, *MANF* and *L3MBTL1* in four of twenty-five. The reported panel was then selected once on all 63
discovery donors and frozen. This notebook asks whether the external performance needs the whole list or only
its stable core.

**Panels.** All 30 genes as reported; the genes reselected in more than 50% of folds; the genes reselected in
more than 70%; and every threshold from 0 to 100% so that no single cut-off is doing the work.

**A null.** For each panel size, 200 random panels drawn from the 5,622 measured genes, scored the same way.
Without it, "fourteen genes also reach 0.8" says nothing.

**What is fixed.** Fold frequencies come from the discovery donors alone and were computed when the panel was
selected. Every forest here is trained on discovery donors only and applied once to each external cohort.

In [ ]:
import os, io, re, gzip, glob, json, time, tarfile, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd
import joblib
from scipy.stats import rankdata, spearmanr, binomtest
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, balanced_accuracy_score
warnings.filterwarnings("ignore")
OUT = Path("/kaggle/working"); GEO = OUT / "geo"; GEO.mkdir(exist_ok=True)
def find_any(pattern, key):
    hits = sorted((h for h in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) if key in h), key=len)
    if not hits:
        raise FileNotFoundError(f"{pattern} ({key})")
    return hits[0]
t0 = time.time()
def log(m): print(f"[{time.time() - t0:5.0f}s] {m}", flush=True)
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_columns", 40)

In [ ]:
DISC = json.loads(r'''{}''')   # the donor-overlap check is not repeated here

## 1. Discovery data, the panel and how often Boruta kept each gene

In [ ]:
cz = np.load(find_any("core_data.npz", "rf-core"), allow_pickle=True)
X, XR, y, DS = cz["X"], cz["XR"], cz["y"].astype(int), cz["ds"].astype(str)
GENES = [str(g) for g in cz["genes"]]
SYM = pd.read_csv(find_any("15_gene_symbol_map.csv", "rf-core")).set_index("gene")["symbol"].to_dict()
PANEL = pd.read_csv(find_any("04_boruta_selected_genes.csv", "boruta-panel")).gene.tolist()
SH = pd.read_csv(find_any("05_boruta_shap_importance.csv", "boruta-panel")).set_index("gene")
FREQ_COL = "boruta_fold_frequency"
FREQ = SH[FREQ_COL].astype(float).to_dict()
assert all(g in FREQ for g in PANEL), "a panel gene has no fold frequency"
PI = [GENES.index(g) for g in PANEL]
print(f"{len(PANEL)} panel genes; fold frequency column '{FREQ_COL}'")
print(pd.Series({SYM.get(g, g): FREQ[g] for g in PANEL}).sort_values(ascending=False).to_string())

In [ ]:
def oob_threshold(s, yt):
    u = np.unique(np.round(s, 6)); cuts = np.concatenate([[-np.inf], (u[:-1] + u[1:]) / 2, [np.inf]])
    acc = [accuracy_score(yt, (s > c).astype(int)) for c in cuts]
    best = np.flatnonzero(np.isclose(acc, max(acc)))
    return float(cuts[best[len(best) // 2]])

def panel_forest(idx, seeds=5):
    """The reported panel procedure on any set of gene columns: five forests, averaged, threshold from OOB."""
    rfs = [RandomForestClassifier(n_estimators=1000, max_features="sqrt", class_weight="balanced", oob_score=True,
                                  random_state=42 + k, n_jobs=-1).fit(X[:, idx], y) for k in range(seeds)]
    thr = oob_threshold(np.mean([m.oob_decision_function_[:, 1] for m in rfs], axis=0), y)
    oof = np.mean([m.oob_decision_function_[:, 1] for m in rfs], axis=0)
    return {"rfs": rfs, "thr": thr, "idx": list(idx), "oob_auc": float(roc_auc_score(y, oof))}

CUTS = [round(c, 2) for c in np.arange(0.0, 1.01, 0.04)]
SUBSETS = {}
for c in CUTS:
    idx = [GENES.index(g) for g in PANEL if FREQ[g] > c]
    if len(idx) >= 3:
        SUBSETS[c] = idx
SUBSETS[-1.0] = PI                                   # the reported panel, all 30 genes
print("panel sizes by threshold:", {c: len(v) for c, v in sorted(SUBSETS.items())})
FOR = {c: panel_forest(idx) for c, idx in SUBSETS.items()}
print("discovery out-of-bag AUC:", {c: round(FOR[c]["oob_auc"], 3) for c in sorted(FOR)})

## 2. The eight bulk cohorts, parsed exactly as in the external validation

In [ ]:
def geo_url(acc, sub, f): return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{acc[:-3]}nnn/{acc}/{sub}/{f}"
def fetch(acc, sub, f):
    dest = GEO / f
    for k in range(6):
        try:
            if not dest.exists():
                urllib.request.urlretrieve(geo_url(acc, sub, f), dest)
            return dest
        except Exception as exc:
            print("  retry", k + 1, type(exc).__name__); dest.unlink(missing_ok=True); time.sleep(10)
    raise RuntimeError(f"could not download {f}")
def read_matrix(path):
    op = gzip.open if str(path).endswith(".gz") else open
    lines = op(path, "rt", encoding="utf-8", errors="replace").read().split("\n")
    meta = {}
    for l in lines:
        if l.startswith("!Sample_"):
            k = l.split("\t")[0]; meta.setdefault(k, []).append([v.strip().strip('"') for v in l.split("\t")[1:]])
    b0 = next((i for i, l in enumerate(lines) if "!series_matrix_table_begin" in l), None)
    b1 = next((i for i, l in enumerate(lines) if "!series_matrix_table_end" in l), None)
    M = None
    if b0 is not None and b1 - b0 > 2:
        M = pd.read_csv(io.StringIO("\n".join(lines[b0 + 1:b1])), sep="\t", index_col=0).apply(pd.to_numeric, errors="coerce")
    return meta, M
def series_matrix(acc, fname=None): return read_matrix(fetch(acc, "matrix", fname or f"{acc}_series_matrix.txt.gz"))
def chars(meta, *keys):
    """One characteristic across samples, whichever characteristics row holds it (first key found)."""
    out = [None] * len(meta["!Sample_geo_accession"][0])
    for row in meta.get("!Sample_characteristics_ch1", []):
        for i, v in enumerate(row):
            for key in keys:
                if out[i] is None and v.lower().startswith(key.lower() + ":"):
                    out[i] = v.split(":", 1)[1].strip()
    return out
def gconvert(ids, target, batch=2500):
    out = {}; ids = list(ids)
    for s in range(0, len(ids), batch):
        body = {"organism": "hsapiens", "target": target, "query": ids[s:s + batch]}
        for k in range(6):
            try:
                req = urllib.request.Request("https://biit.cs.ut.ee/gprofiler/api/convert/convert/", data=json.dumps(body).encode(),
                                             headers={"Content-Type": "application/json"})
                with urllib.request.urlopen(req, timeout=300) as fh:
                    for rec in json.loads(fh.read())["result"]:
                        c = rec.get("converted")
                        if c and c not in ("None", "N/A"):
                            out.setdefault(rec["incoming"], set()).add(c)
                break
            except Exception as exc:
                print("  g:Convert retry", k + 1, type(exc).__name__); time.sleep(10)
    return out
MARKERS = ["TH", "SLC6A3", "SLC18A2", "DDC", "KCNJ6", "ALDH1A1", "NR4A2", "EN1"]        # dopamine-neuron content
SEXG = ["XIST", "RPS4Y1", "DDX3Y", "KDM5D", "UTY", "EIF1AY"]                             # sex, from expression
def array_genes(M, namespace):
    """Probe matrix -> gene matrix (each gene's highest-mean probe), log2 unless already logged.
    HG-U133A probe sets all exist, under the same IDs, on HG-U133 Plus 2, so both use the Plus 2 mapping."""
    L = np.log2(M.clip(lower=1)) if np.nanmax(M.values) > 50 else M.copy()
    pm = L.mean(1)
    def best(ids):
        conv = gconvert(ids, namespace)
        return {g: max((p for p in ps if p in L.index), key=lambda p: pm[p]) for g, ps in conv.items() if any(p in L.index for p in ps)}
    bg, bm = best(GENES), best(MARKERS + SEXG)
    G = pd.DataFrame({g: L.loc[p].to_numpy(float) for g, p in bg.items()}, index=L.columns).T
    Mk = pd.DataFrame({s: L.loc[p].to_numpy(float) for s, p in bm.items()}, index=L.columns).T
    return G, Mk
def sex_from_expression(Mk):
    """Female = low Y-chromosome genes and high XIST; split at the widest gap of the combined score."""
    z = lambda D: ((D.T - D.T.mean()) / D.T.std(ddof=1).clip(lower=0.05)).T
    yg = [g for g in SEXG[1:] if g in Mk.index]
    s = z(Mk.loc[yg]).mean(0).to_numpy() - (z(Mk.loc[["XIST"]]).mean(0).to_numpy() if "XIST" in Mk.index else 0)
    v = np.sort(s); i = int(np.argmax(np.diff(v)))
    return (s < (v[i] + v[i + 1]) / 2).astype(int)
def meta_sex(vals):
    return [None if v is None else (1 if v.strip().lower().startswith("f") else 0) for v in vals]
COH = {}
def add(name, G, Mk, yv, donors, note, sex=None, stage=None):
    yv, donors = np.asarray(yv, int), np.asarray(donors, str)
    COH[name] = dict(G=G, Mk=Mk, y=yv, donor=donors, note=note, sex_meta=sex if sex is not None else [None] * len(yv),
                     sex_expr=sex_from_expression(Mk), stage=stage if stage is not None else ["PD" if v else "control" for v in yv])
    log(f"{name}: {len(yv)} people ({int((yv == 0).sum())} control, {int(yv.sum())} PD); {G.shape[0]:,}/{len(GENES):,} genes; "
        f"{sum(m in Mk.index for m in MARKERS)}/8 neuron markers  [{note}]")

In [ ]:
# ---------------- GSE7621 (HG-U133 Plus 2) - the same file as pd-lcm-rf-external ----------------
meta, M = read_matrix(find_any("GSE7621_series_matrix.txt*", "externalvalidation2"))
tit = meta["!Sample_title"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE7621", G, Mk, [0 if "normal" in t.lower() else 1 for t in tit], tit, "Plus 2", sex=meta_sex(chars(meta, "gender", "sex")))

# ---------------- GSE20292 (HG-U133A); donor numbers shared with discovery GSE20141 ----------------
meta, M = series_matrix("GSE20292")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state")
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20292", G, Mk, [0 if d.lower().startswith("control") else 1 for d in dx], [t.split()[0] for t in tit], "U133A",
    sex=meta_sex(chars(meta, "gender")))

# ---------------- GSE20163 (HG-U133A) ----------------
meta, M = series_matrix("GSE20163")
tit = meta["!Sample_title"][0]; src = meta["!Sample_source_name_ch1"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20163", G, Mk, [0 if "control" in s.lower() else 1 for s in src], [t.split("_")[0] for t in tit], "U133A")

# ---------------- GSE20164 (HG-U133A) ----------------
meta, M = series_matrix("GSE20164")
tit = meta["!Sample_title"][0]; src = meta["!Sample_source_name_ch1"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20164", G, Mk, [0 if "control" in s.lower() else 1 for s in src], tit, "U133A", sex=meta_sex(chars(meta, "gender")))

# ---------------- GSE8397 (HG-U133A chip): lateral and medial nigra averaged per case; frontal cortex left out ----------------
meta, M = series_matrix("GSE8397", "GSE8397-GPL96_series_matrix.txt.gz")
tit = meta["!Sample_title"][0]; ag = chars(meta, "age")
keep = [i for i, t in enumerate(tit) if "substantia nigra" in t.lower()]
def case_no(t):
    m = re.search(r"case\s*(\d+)", t, re.I) or re.search(r"(\d+)[^\d]*-\s*[AB]\s*chip", t, re.I) or re.search(r"(\d+)", t)
    return int(m.group(1))
print("   GSE8397 nigra titles:", [tit[i] for i in keep])
case = [("PD" if "parkinson" in tit[i].lower() else "C") + "-" + str(case_no(tit[i])) for i in keep]
sexc = {c: (1 if "gender: f" in (ag[i] or "").lower() else 0) for c, i in zip(case, keep)}
L = M.iloc[:, keep].copy(); L.columns = case
L = L.T.groupby(level=0).mean().T                                          # one profile per person
print("   GSE8397 nigra samples per case:", pd.Series(case).value_counts().value_counts().to_dict())
G, Mk = array_genes(L, "AFFY_HG_U133_PLUS_2")
add("GSE8397", G, Mk, [1 if c.startswith("PD") else 0 for c in L.columns], list(L.columns), "U133A, lateral+medial SN averaged",
    sex=[sexc[c] for c in L.columns])

# ---------------- GSE49036 (Plus 2), Netherlands Brain Bank: control vs PD; ILBD kept for the early-stage test ----------------
meta, M = series_matrix("GSE49036")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state"); braak = chars(meta, "braak stage")
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
lab = np.array([0 if d.lower() == "control" else (1 if "parkinson" in d.lower() else 2) for d in dx])   # 2 = incidental Lewy body
print("   GSE49036 diagnosis x Braak:", pd.crosstab(np.array(dx), np.array(braak)).to_dict())
m = lab < 2
add("GSE49036", G.loc[:, m], Mk.loc[:, m], lab[m], np.array(tit)[m], "Plus 2, NBB")
EARLY_RAW = dict(G=G, Mk=Mk, lab=lab, braak=np.array(braak), donor=np.array(tit))

In [ ]:
# ---------------- GSE114517 (RNA-seq counts per sample; substantia nigra only; PD with dementia) ----------------
meta, _ = series_matrix("GSE114517")
gsm = meta["!Sample_geo_accession"][0]; tit = meta["!Sample_title"][0]
tissue = chars(meta, "tissue"); st = chars(meta, "subject status", "disease state"); sx = chars(meta, "gender", "sex")
sn = [i for i, t in enumerate(tissue) if t and "substantia" in t.lower()]
cols = {}
with tarfile.open(fetch("GSE114517", "suppl", "GSE114517_RAW.tar")) as tf:
    for mem in tf.getmembers():
        g = mem.name.split("_")[0]
        if g in {gsm[i] for i in sn}:
            raw = tf.extractfile(mem).read()
            txt = (gzip.decompress(raw) if mem.name.endswith(".gz") else raw).decode()
            s = pd.read_csv(io.StringIO(txt), sep="\t", header=None, index_col=0)[1]
            s.index = s.index.astype(str).str.split(".").str[0]
            cols[g] = s.groupby(level=0).sum()
C = pd.DataFrame(cols)[[gsm[i] for i in sn]].fillna(0)
C = C[~C.index.str.startswith("__")]
print(f"   GSE114517: {C.shape[1]} nigra libraries, median {np.median(C.sum(0)) / 1e6:.1f} M counted reads")
LC = np.log2(C / C.sum(0) * 1e6 + 1)
SYM2E = {s: sorted(v) for s, v in gconvert(MARKERS + SEXG, "ENSG").items()}
def rna_markers(LG):
    return pd.DataFrame({s: LG.loc[[e for e in es if e in LG.index]].sum(0).to_numpy(float) for s, es in SYM2E.items()
                         if any(e in LG.index for e in es)}, index=LG.columns).T
add("GSE114517", LC.reindex([g for g in GENES if g in LC.index]), rna_markers(LC),
    [0 if "control" in st[i].lower() else 1 for i in sn], [re.search(r"\[(.*?)\]", tit[i]).group(1) for i in sn],
    "RNA-seq, PD with dementia", sex=meta_sex([sx[i] for i in sn]))

# ---------------- GSE168496 (RNA-seq, transcript level; Netherlands Brain Bank IDs) ----------------
meta, _ = series_matrix("GSE168496")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state")
T = pd.read_csv(fetch("GSE168496", "suppl", "GSE168496_all_samples_preprocessed_data.tsv.gz"), sep="\t", index_col=0)
T.index = T.index.astype(str).str.split(".").str[0]
T = T.groupby(level=0).sum()[tit]
def tx_to_gene(ids):
    conv = gconvert(ids, "ENST", batch=1500)
    return pd.DataFrame({g: T.loc[[t for t in ts if t in T.index]].sum(0).to_numpy(float) for g, ts in conv.items()
                         if any(t in T.index for t in ts)}, index=T.columns).T
GT = np.log2(tx_to_gene(GENES) + 1)
MT = np.log2(tx_to_gene(MARKERS + SEXG) + 1)
print(f"   GSE168496: {T.shape[0]:,} transcripts, {T.shape[1]} people; column sums {T.sum(0).min():,.0f}-{T.sum(0).max():,.0f}")
add("GSE168496", GT, MT, [0 if "control" in d.lower() else 1 for d in dx], tit, "RNA-seq, NBB", sex=meta_sex(chars(meta, "gender")))
COV = pd.DataFrame([{"cohort": n, "platform": c["note"], "people": len(c["y"]), "control": int((c["y"] == 0).sum()), "PD": int(c["y"].sum()),
                     "discovery_genes_measured": c["G"].shape[0], "share": c["G"].shape[0] / len(GENES),
                     "panel_genes_measured": int(np.isin(PANEL, c["G"].index).sum()),
                     "neuron_markers": int(np.isin(MARKERS, c["Mk"].index).sum())} for n, c in COH.items()])
COV.to_csv(OUT / "cohorts.csv", index=False); print(COV.round(3).to_string(index=False))

## 3. Every panel in every cohort

In [ ]:
from scipy.stats import rankdata
zrow = lambda D: ((D.T - D.T.mean()) / D.T.std(ddof=1).clip(lower=0.05)).T
rng = np.random.default_rng(42)

def boot_ci(yv, s, n=4000):
    bs = [roc_auc_score(yv[i], s[i]) for i in (rng.integers(0, len(yv), len(yv)) for _ in range(n)) if len(set(yv[i])) == 2]
    return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

Z = {}
for name, c in COH.items():
    Z[name] = zrow(c["G"]).reindex(GENES).fillna(0.0).T.to_numpy()
    print(f"{name}: {len(c['y'])} donors, {int(c['G'].shape[0])} genes measured")

def score_panel(P, Zc):
    return np.mean([m.predict_proba(Zc[:, P["idx"]])[:, 1] for m in P["rfs"]], axis=0)

rows = []
for c, P in sorted(FOR.items()):
    for name, co in COH.items():
        yv = np.asarray(co["y"], int)
        if len(set(yv)) < 2:
            continue
        s = score_panel(P, Z[name])
        lo, hi = boot_ci(yv, s)
        measured = int(np.isin([GENES[i] for i in P["idx"]], co["G"].index).sum())
        rows.append(dict(threshold=c, n_genes=len(P["idx"]), cohort=name, n=len(yv),
                         genes_measured=measured, auc=float(roc_auc_score(yv, s)), ci_lo=lo, ci_hi=hi,
                         accuracy=float(accuracy_score(yv, (s > P["thr"]).astype(int)))))
BYCOH = pd.DataFrame(rows)
BYCOH.to_csv(OUT / "stability_auc_by_cohort.csv", index=False)
print(BYCOH[BYCOH.threshold.isin([-1.0, 0.48, 0.68])].round(3).to_string(index=False))

In [ ]:
# pooled over cohorts, each donor once, weighted by cohort size
def pooled(d):
    w = d.n / d.n.sum()
    return float((d.auc * w).sum())
POOL = BYCOH.groupby(["threshold", "n_genes"]).apply(pooled).rename("pooled_auc").reset_index()
POOL["genes"] = [", ".join(SYM.get(GENES[i], GENES[i]) for i in FOR[t]["idx"]) for t in POOL.threshold]
POOL["discovery_oob_auc"] = [FOR[t]["oob_auc"] for t in POOL.threshold]
POOL.to_csv(OUT / "stability_pooled_auc.csv", index=False)
print(POOL[["threshold", "n_genes", "discovery_oob_auc", "pooled_auc"]].round(3).to_string(index=False))

## 4. Against random panels of the same size

A panel of fourteen genes that reaches 0.8 is only interesting if fourteen genes drawn at random do not.
Each random panel goes through the identical procedure: five forests on the discovery donors, then scored
once in every cohort and pooled the same way.

In [ ]:
# the null is expensive, so it is run for the three panels the paper reports: all 30, >50% and >70% of folds
SIZES = sorted({len(FOR[t]["idx"]) for t in (-1.0, 0.48, 0.68)})
print("random-panel null for sizes:", SIZES)
NRAND = 200
RAND = {}
for k in SIZES:
    aucs = []
    for b in range(NRAND):
        idx = list(rng.choice(len(GENES), size=k, replace=False))
        P = panel_forest(idx, seeds=1)
        d = pd.DataFrame([dict(n=len(COH[n]["y"]),
                               auc=roc_auc_score(np.asarray(COH[n]["y"], int), score_panel(P, Z[n])))
                          for n in COH if len(set(COH[n]["y"])) > 1])
        aucs.append(float((d.auc * d.n / d.n.sum()).sum()))
    RAND[k] = np.array(aucs)
    print(f"{k:3d} random genes: median pooled AUC {np.median(aucs):.3f}, 95th percentile {np.percentile(aucs, 95):.3f}")
pd.DataFrame({f"size_{k}": v for k, v in RAND.items()}).to_csv(OUT / "stability_random_panels.csv", index=False)

In [ ]:
rows = []
for t in sorted(FOR):
    k = len(FOR[t]["idx"])
    obs = float(POOL.loc[POOL.threshold == t, "pooled_auc"].iloc[0])
    if k not in RAND:                       # sizes outside the three reported panels carry no null
        rows.append(dict(threshold=t, n_genes=k, pooled_auc=obs, random_median=np.nan,
                         random_p95=np.nan, p_vs_random=np.nan))
        continue
    null = RAND[k]
    rows.append(dict(threshold=t, n_genes=k, pooled_auc=obs,
                     random_median=float(np.median(null)), random_p95=float(np.percentile(null, 95)),
                     p_vs_random=float((np.sum(null >= obs) + 1) / (len(null) + 1))))
VS = pd.DataFrame(rows)
VS.to_csv(OUT / "stability_vs_random.csv", index=False)
print(VS.round(3).to_string(index=False))

REPORTED = VS[VS.threshold == -1.0].iloc[0]
BEST = VS[(VS.threshold >= 0) & VS.p_vs_random.notna()].sort_values("pooled_auc", ascending=False).iloc[0]
SUMMARY = dict(reported=dict(n_genes=int(REPORTED.n_genes), pooled_auc=float(REPORTED.pooled_auc),
                             p_vs_random=float(REPORTED.p_vs_random)),
               best_stable=dict(threshold=float(BEST.threshold), n_genes=int(BEST.n_genes),
                                pooled_auc=float(BEST.pooled_auc), p_vs_random=float(BEST.p_vs_random),
                                genes=[SYM.get(GENES[i], GENES[i]) for i in FOR[BEST.threshold]["idx"]]),
               at_50=VS[VS.threshold == 0.48].to_dict("records"),
               at_70=VS[VS.threshold == 0.68].to_dict("records"),
               fold_frequency={SYM.get(g, g): FREQ[g] for g in PANEL})
json.dump(SUMMARY, open(OUT / "stability_summary.json", "w"), indent=1)
print(json.dumps({k: v for k, v in SUMMARY.items() if k != "fold_frequency"}, indent=1))

## 5. Reading this honestly

If the stable subset matches the full panel, the reported 30 genes are carrying about as much as their core
does, and the unstable tail is candidate biology rather than signal. If it does better, the tail is noise
worth dropping. If it does worse, the tail matters and the panel should stay whole. The random-panel null
decides how much of any of it is the panel and how much is simply fitting thirty numbers to 63 people.